# 04 · Metingen publiceren met MQTT

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een database bewaart gegevens. Om die gegevens aan een ander programma door te geven, moet een programma ze lezen en versturen. Je gebruikt daarvoor Python in Codespaces. Mosquitto op je eigen server verdeelt de berichten over de aangesloten ontvangers.

Je begint met een klein zelfgemaakt bericht. Daarna verstuur je twintig bestaande meetrecords uit jouw werkdatabase en onderzoek je wat er gebeurt wanneer je dezelfde selectie opnieuw aanbiedt. Zo kun je elk onderdeel van de uitwisseling bekijken voordat je later een volledige beginset deelt.

**Nodig:** je eigen branch en Codespace uit notebook 01, de aangevulde werkdatabase uit notebook 02, je ingerichte server uit notebook 03, `gegevens/werk/server.json`, `gegevens/werk/ca.crt` en het MQTT-wachtwoord van account **eigenaar**. Gebruik de Python-kernel van de Codespace. De Python-pakketten zijn al opgenomen in de werkomgeving van de repository.

Voer de cellen van boven naar beneden uit. De wachtwoordvragen verschijnen tijdens het uitvoeren. Tussen de cellen noteer je eigen waarnemingen en pas je een oefenbericht aan.

De bewaarde uitvoer toont een voorbeeld met deelnemerscode `proef`. Tijdens jouw uitvoering verschijnen jouw eigen instellingen en resultaten.

</div>

## 1. Je werkplek en server hervatten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

De rol uit notebook 01 bepaalt welke werkdatabase wordt geopend. Bij gewichten is dat `asserva.db`, bij voederopname `schauer.db`. De twee bronnen hebben verschillende meetkolommen. De aangeleverde code kiest `VBW` met eenheid kg of `VFI` met eenheid g op basis van jouw rol.

De onderstaande cel controleert de opgeslagen instellingen en de aanwezigheid van je werkbestanden. Bewaar een ontbrekende werkdatabase eerst via notebook 02. Maak geen leeg bestand met dezelfde naam.

</div>

In [1]:
# Tijdelijke hulpcel, verwijderd bij het genereren van de studentenversie.
_DOCENT_BRANCH_OVERRIDE = "student/proefdocent"

In [2]:
from pathlib import Path
from contextlib import closing
from getpass import getpass
from datetime import datetime, timezone
import subprocess
import sys
import json
import re
import ipaddress

root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if root.returncode:
    raise RuntimeError("Open notebook 04 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(root.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch) and "_DOCENT_BRANCH_OVERRIDE" in globals():
    branch = _DOCENT_BRANCH_OVERRIDE
instellingen = json.loads((hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json").read_text(encoding="utf-8"))
deelnemer = instellingen["deelnemerscode"]
if not re.fullmatch(r"[a-z0-9][a-z0-9_-]{0,62}", deelnemer) or branch != "student/" + deelnemer or instellingen.get("branch") != branch:
    raise ValueError("Controleer je eigen branch en werkplekcontrole uit notebook 01.")
rol = instellingen["rol"]
bestanden = {"gewichten": "asserva.db", "voederopname": "schauer.db"}
if rol not in bestanden or instellingen.get("bronbestand") != bestanden[rol]:
    raise ValueError("Rol en databank komen niet overeen. Controleer notebook 01.")
werkmap = hoofdmap / "gegevens/werk"
werkdatabase = werkmap / bestanden[rol]
server = json.loads((werkmap / "server.json").read_text(encoding="utf-8"))
if server.get("deelnemerscode") != deelnemer:
    raise ValueError("Dit serverbestand hoort bij een andere deelnemer.")
serveradres = str(ipaddress.IPv4Address(server["ipv4"]))
ca = werkmap / "ca.crt"
for pad in (werkdatabase, ca):
    if not pad.is_file():
        raise FileNotFoundError(f"Ontbreekt: {pad.relative_to(hoofdmap)}. Hervat notebook 02 of 03.")
sys.path.insert(0, str(hoofdmap / "programmatuur"))
from publiceren import MqttSessie, lees_metingen, meettopic, publiceer_selectie
print("Deelnemer:", deelnemer, "| Rol:", rol)
print("Werkdatabase:", werkdatabase.relative_to(hoofdmap))
print("SSH-opdracht:", f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

Deelnemer: proef | Rol: gewichten
Werkdatabase: gegevens/werk/asserva.db
SSH-opdracht: ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Terminal), vervolgens op je DigitalOcean-server (SSH-terminal, als beheer)

1. Open **Terminal → New Terminal** en voer de getoonde SSH-opdracht uit. Gebruik je beheerwachtwoord uit notebook 03.
2. Start Mosquitto met onderstaande opdrachten. Na een serverherstart staat deze broker bewust uit. Als hij al draait, laat `start` hem doorwerken.
3. Controleer of `active (running)` verschijnt. Keer daarna terug naar notebook 04. De broker blijft draaien wanneer je de SSH-terminal sluit.

</div>

```bash
sudo systemctl start mosquitto
sudo systemctl status mosquitto --no-pager
```

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Let op</strong><br>Het SSH-wachtwoord dient voor serverbeheer. In de volgende cellen gebruik je het afzonderlijke MQTT-wachtwoord van <strong>eigenaar</strong>. Typ het alleen in het verborgen invoerveld. Sla het niet op in een cel, antwoord, screenshot of Git-commit.</div>

## 2. Wat wordt er uitgewisseld?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Een **publisher** verstuurt een bericht. Een **subscriber** vraagt berichten op een bepaald topic aan. Beide zijn MQTT-clients en verbinden met een **broker**. De broker ontvangt een publicatie en geeft die door aan aangesloten clients met een passend abonnement. Eén client kan zowel publiceren als ontvangen.

Een MQTT-bericht heeft een **topic** en een **payload**. Het topic benoemt de berichtenstroom, bijvoorbeeld `practicum/voornaam_achternaam/metingen/gewichten`. De payload is de eigenlijke inhoud. Hier kiezen we JSON als formaat voor die inhoud. MQTT schrijft JSON niet voor.

Een topic is geen SQL-tabel. Het is een tekst waarop de broker abonnementen vergelijkt. De broker weet daardoor nog niet hoe een meetwaarde in een database moet worden opgeslagen. In deze oefening bekijkt een tijdelijk abonnement de ontvangst in het notebook.

</div>

| Onderdeel | Plaats | Functie in deze oefening |
|---|---|---|
| Werkdatabase | Codespace | Bewaart de meetrecords en hokmetadata |
| Publicerende Python-code | Codespace | Leest records en maakt berichten |
| Mosquitto | Eigen Droplet | Verdeelt berichten op basis van topics |
| Tijdelijk abonnement | Codespace | Laat ontvangen inhoud zien |
| Permanente ontvanger | Volgende oefening | Slaat ontvangen metingen in SQLite op |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Waarom is alleen Mosquitto starten onvoldoende om je database te delen?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):**

Voeg hier je antwoord in. Benoem de taak van de publicerende code en die van de broker.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 1**

Mosquitto leest de SQLite-database niet. Python moet records uitlezen en als berichten publiceren. Een ontvangprogramma moet ontvangen berichten vervolgens verwerken of opslaan.

</div>

## 3. Zelf een JSON-bericht samenstellen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

**JSON** beschrijft gegevens met sleutel-waardeparen. Een object staat tussen accolades. Een tekstwaarde staat tussen dubbele aanhalingstekens. Getallen staan zonder aanhalingstekens en gebruiken een punt als decimaalteken. Een JSON-object kan ook een ander object of een lijst bevatten.

In Python maken we eerst een `dict`, een verzameling sleutel-waardeparen. `json.dumps` zet die om naar JSON-tekst. De woordenboeken lijken op elkaar, maar het zijn verschillende dingen: een Python-object bestaat in het programma, JSON-tekst kan over een verbinding worden verstuurd. Een ontbrekende waarde wordt in Python `None` en in JSON `null`. Dit is iets anders dan het getal nul.

Je oefenbericht beschrijft de oppervlakte van hok 1. Het tijdstip geeft aan wanneer je het oefenbericht hebt gemaakt, niet wanneer een big is gewogen. De eenheid hoort bij de waarde en wordt daarom expliciet meegegeven.

</div>

In [3]:
oefenbericht = {
    "soort": "oefening",
    "hok": "1",
    "eigenschap": "oppervlakte",
    "waarde": 8.0,
    "eenheid": "m2",
    "gemaakt_op": datetime.now(timezone.utc).isoformat(),
}
json_voorbeeld = json.dumps(oefenbericht, ensure_ascii=False, indent=2, allow_nan=False)
print(json_voorbeeld)
print("Python-type:", type(oefenbericht).__name__)
print("Type na omzetting:", type(json_voorbeeld).__name__)

{
  "soort": "oefening",
  "hok": "1",
  "eigenschap": "oppervlakte",
  "waarde": 8.0,
  "eenheid": "m2",
  "gemaakt_op": "2026-11-18T12:54:50.300480+00:00"
}
Python-type: dict
Type na omzetting: str


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Welke onderdelen van dit bericht zijn tekst, welke zijn getallen en waarom staat het hoknummer tussen aanhalingstekens?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):**

Voeg hier je antwoord in. Gebruik minstens één voorbeeld van elk type uit jouw uitvoer.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 2**

De sleutelwoorden en de waarden van soort, hok, eigenschap, eenheid en gemaakt_op zijn tekst. Waarde is een getal. Het hoknummer is een identificatiecode en blijft tekst, zoals PEN en hok in de database. Het tijdstip is ook tekst en bevat hier een expliciete UTC-offset.

</div>

## 4. Publiceren en de ontvangst bekijken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Je gebruikt een apart oefentopic onder `controle/`, zodat dit bericht niet tussen echte metingen terechtkomt. De cel maakt een beveiligde verbinding, vraagt een abonnement aan en wacht op de bevestiging van de broker. Pas daarna wordt gepubliceerd. Een willekeurige wachttijd zou geen betrouwbare controle zijn dat het abonnement klaarstaat.

`MqttSessie` is aangeleverde ondersteunende code. Het `with`-blok opent de verbinding en sluit die na afloop weer, ook bij een fout. Het CA-bestand uit notebook 03 controleert het servercertificaat. Het wachtwoord wordt voor elke korte proef verborgen gevraagd en niet in een bestand geschreven.

</div>

In [4]:
oefentopic = f"practicum/{deelnemer}/controle/json"
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: "), abonnement=oefentopic) as verbinding:
    verbinding.verstuur(oefentopic, oefenbericht)
    ontvangen_oefening = verbinding.lees()
assert ontvangen_oefening["payload"] == oefenbericht
print("Ontvangen topic:", ontvangen_oefening["topic"])
print(json.dumps(ontvangen_oefening["payload"], ensure_ascii=False, indent=2))

Ontvangen topic: practicum/proef/controle/json
{
  "eenheid": "m2",
  "eigenschap": "oppervlakte",
  "gemaakt_op": "2026-11-18T12:54:50.300480+00:00",
  "hok": "1",
  "soort": "oefening",
  "waarde": 8.0
}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>Je ziet hetzelfde topic en dezelfde inhoud terug. Dit is daadwerkelijk ontvangen MQTT-verkeer. De vergelijking controleert de ontvangen inhoud, niet alleen of de publicatieopdracht zonder fout eindigde.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Doe-opdracht:** maak een kopie van `oefenbericht` met `dict(oefenbericht)` en noem die `aangepast`. Verander alleen `waarde` naar `9.0`. Dit is een tijdelijke oefenwaarde. De werkelijke hokoppervlakte in je database blijft 8 m². Voer daarna de ontvangstcel uit en noteer welke waarde terugkomt.

</div>

In [5]:
# Maak een kopie: aangepast = dict(oefenbericht)
# Vul daarna in: aangepast["waarde"] = ...

In [6]:
aangepast = dict(oefenbericht)
aangepast["waarde"] = 9.0

In [7]:
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: "), abonnement=oefentopic) as verbinding:
    verbinding.verstuur(oefentopic, aangepast)
    ontvangen_aanpassing = verbinding.lees()
assert ontvangen_aanpassing["payload"] == aangepast
print("Ontvangen oefenwaarde:", ontvangen_aanpassing["payload"]["waarde"])
print("Eerste oefenwaarde:", oefenbericht["waarde"])

Ontvangen oefenwaarde: 9.0
Eerste oefenwaarde: 8.0


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Welke waarde ontving je na de aanpassing? Is daardoor de oppervlakte in hokken gewijzigd?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):**

Voeg hier je antwoord in. Maak onderscheid tussen een Python-object, een MQTT-bericht en een opgeslagen tabelrij.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 3**

De ontvangen oefenwaarde is 9.0. De oorspronkelijke oefenwaarde is 8.0 en de hokoppervlakte in SQLite blijft 8 m². Alleen het tijdelijke Python-object is aangepast en gepubliceerd. Er is geen SQL-opdracht uitgevoerd om de database te wijzigen.

</div>

## 5. Een echt meetrecord lezen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Nu lees je uit de werkdatabase van notebook 02. We kiezen de eerste twintig records, gesorteerd op `DATE` en `ROW_ID`. Dit is een kleine technische selectie om berichten te volgen. Ze is niet bedoeld als representatieve selectie voor een berekening van voederconversie.

De functie `lees_metingen` opent je database alleen om te lezen. Ze maakt voor elke rij een bericht met herkenbare Nederlandse velden. Het volledige oorspronkelijke record blijft daarnaast aanwezig onder `bronrecord`, inclusief kwaliteitsvlaggen. De bijbehorende hokmetadata uit notebook 02 staan onder `hokmetadata`.

De combinatie **bron en record_id** identificeert een meting. De lange `ROW_ID` blijft tekst. Bij herpublicatie krijgt dezelfde bronrij dezelfde identiteit. De deelnemer geeft aan wie publiceert en verandert de identiteit van de oorspronkelijke meting niet.

</div>

In [8]:
aantal_in_selectie = 20
selectie = list(lees_metingen(werkdatabase, rol, deelnemer, limiet=aantal_in_selectie))
if len(selectie) != aantal_in_selectie:
    raise ValueError("De werkdatabase bevat minder records dan de gevraagde selectie.")
topic_metingen = meettopic(deelnemer, rol)
eerste = selectie[0]
print("Topic:", topic_metingen)
print("Geselecteerde records:", len(selectie))
print(json.dumps(eerste, ensure_ascii=False, indent=2, allow_nan=False))

Topic: practicum/proef/metingen/gewichten
Geselecteerde records: 20
{
  "schema_versie": 1,
  "deelnemer": "proef",
  "bron": "asserva",
  "meettype": "gewichten",
  "record_id": "897280101692878400000000000",
  "dier_id": "89728010",
  "hok": "3",
  "tijd": {
    "start": "2023-08-24T12:00:00",
    "einde": "2023-08-24T12:00:01"
  },
  "tijdzone": "onbevestigd; vermoedelijk Europe/Brussels",
  "meetveld": "VBW",
  "waarde": 6.68,
  "eenheid": "kg",
  "hokmetadata": {
    "hok": "3",
    "gebouw": "M5",
    "afdeling": 1,
    "oppervlakte_m2": 8.0
  },
  "bronrecord": {
    "ROW_ID": "897280101692878400000000000",
    "EID": "89728010",
    "PEN": "3",
    "TREAT": "C",
    "GENDER": "0",
    "DATE": "2023-08-24",
    "TIMEstart": "12:00:00",
    "TIMEstop": "12:00:01",
    "TIMESTAMPstart": "20230824120000",
    "TIMESTAMPstop": "20230824120001",
    "PROCESSTIME": 0.0,
    "TRIALDAY": 0.5,
    "DTMARKER": 0.5,
    "DTMARKER_INT": 0.0,
    "VBW": 6.68,
    "DATAORIGIN": "allocation",


| Veld | Betekenis |
|---|---|
| `schema_versie` | Versie van de afgesproken berichtstructuur |
| `bron`, `record_id` | Stabiele combinatie die de bronrij identificeert |
| `deelnemer` | Deelnemerscode van de publicerende client |
| `meettype`, `meetveld` | Gewichten met VBW, of voederopname met VFI |
| `dier_id`, `hok` | Diercode EID en hokcode PEN, als tekst |
| `tijd.start`, `tijd.einde` | Geformatteerde bezoekstijdstippen, zonder onbevestigde UTC-offset |
| `waarde`, `eenheid` | VBW in kg of VFI in g per bezoek |
| `hokmetadata` | Gebouw, afdeling en oppervlakte uit jouw tabel hokken |
| `bronrecord` | Volledige oorspronkelijke rij met ongewijzigde veldnamen en waarden |

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Uitleg</strong><br>Asserva bewaart bijvoorbeeld <code>20230824120000</code>, Schauer gebruikt datum-tijdtekst. De code zet beide om naar een leesbaar formaat zoals <code>2023-08-24T12:00:00</code>. De bron bevat geen tijdzone-offset. Daarom wordt geen <code>Z</code> of UTC-offset toegevoegd. De vermoedelijke Belgische tijdzone blijft als onzekerheid vermeld. Dit verschilt van het huidige oefentijdstip, waarvan de UTC-offset wel bekend is.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Zoek het eerste getoonde record terug in DB Browser op je laptop. Vergelijk ROW_ID, EID, PEN en VBW of VFI. Welke bronkolom levert jouw waarde en wat is de eenheid?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

Voeg hier je antwoord in. Gebruik Browse Data en het filter bij ROW_ID. Noteer jouw record_id en meetwaarde, zodat je vergelijking controleerbaar is.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 4**

Bij de gewichtsrol komt waarde uit VBW en is de eenheid kg. Bij voederopname komt waarde uit VFI en is de eenheid g per bezoek. record_id, dier_id en hok komen rechtstreeks uit ROW_ID, EID en PEN. De eerste bronidentificatie blijft exact gelijk, inclusief alle cijfers.

</div>

## 6. Een meetrecord via de broker volgen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Het topic met `metingen/` scheidt de echte records van de eerdere oefenberichten. Eerst abonneer je op jouw meettopic. Daarna verstuur je de eerste bronrij en lees je haar terug. Vergelijk de ontvangen identificatie en meetwaarde met de vorige stap.

Dit meekijkbericht schrijft nog niets in de verzendadministratie van de volgende stap. Daardoor kun je hetzelfde record zo vaak bekijken als nodig. Een toekomstige ontvanger moet herhalingen herkennen aan bron en record_id.

</div>

In [9]:
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: "), abonnement=topic_metingen) as verbinding:
    verbinding.verstuur(topic_metingen, eerste)
    ontvangen_meting = verbinding.lees()
assert ontvangen_meting["payload"] == eerste
print("Ontvangen topic:", ontvangen_meting["topic"])
print("Bron en record:", ontvangen_meting["payload"]["bron"], ontvangen_meting["payload"]["record_id"])
print("Waarde:", ontvangen_meting["payload"]["waarde"], ontvangen_meting["payload"]["eenheid"])

Ontvangen topic: practicum/proef/metingen/gewichten
Bron en record: asserva 897280101692878400000000000
Waarde: 6.68 kg


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een abonnement kan ook een filter bevatten. `+` staat voor één niveau en `#` voor alle volgende niveaus. Het filter `practicum/jouw_deelnemerscode/metingen/+` past daardoor op beide meettypen onder jouw deelnemerscode. Het maakt een andere bron niet vanzelf beschikbaar. De juiste publisher moet op jouw broker publiceren en het account moet leesrechten hebben.

**Doe-opdracht:** vervang alleen `abonnement=topic_metingen` in de vorige cel door `abonnement=f"practicum/{deelnemer}/metingen/+"`. Voer de cel opnieuw uit en controleer of dezelfde meting aankomt. Laat het publicatietopic ongewijzigd. Een wildcard hoort in het abonnement, niet in het topic waarop je publiceert.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Waarom ontvangt het filter met + jouw bericht, terwijl een abonnement op controle/json dat meetbericht niet ontvangt?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

Voeg hier je antwoord in. Schrijf jouw concrete topic en het passende filter op.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 5**

Het teken + vervangt precies het laatste niveau met de rolnaam in het meettopic. controle/json is een ander topic en komt niet overeen met metingen/gewichten of metingen/voederopname. De broker vergelijkt topicnamen en filters, niet de betekenis van de JSON-velden.

</div>

## 7. Nieuwe records versturen en bewust herpubliceren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04)

Wanneer je later extra records aan je werkdatabase toevoegt, wil je kunnen zien welke records nog niet zijn aangeboden. De publicatiecode houdt daarom een kleine **verzendadministratie** bij in `gegevens/werk/publicaties_04.db`. Ze registreert per broker, topic en bronrecord of de broker de publicatie heeft bevestigd. Je werkdatabase blijft ongewijzigd.

Bij **nieuwe records versturen** slaat de code eerder bevestigde records in dezelfde selectie over. Bij **herpubliceren** wordt de hele aangeboden selectie opnieuw verstuurd met dezelfde identificaties. Herpublicatie is nodig wanneer een ontvanger nog niet actief was of berichten heeft gemist.

MQTT **QoS 1** vraagt een bevestiging en kan een bericht meermaals afleveren. Zo'n bevestiging bewijst dat de broker het bericht heeft verwerkt. Ze bewijst niet dat een ontvangprogramma het in een SQL-database heeft opgeslagen. In notebook 05 krijgt die ontvanger een controle op dubbele bronidentificaties.

De onderstaande cellen publiceren alleen de twintig getoonde records. Op een eerste uitvoering verwacht je twintig bevestigingen. Bij hervatten kunnen ze al in de verzendadministratie staan en zie je twintig overgeslagen records.

</div>

In [10]:
administratie = werkmap / "publicaties_04.db"
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: ")) as verbinding:
    eerste_publicatie = publiceer_selectie(verbinding, topic_metingen, selectie, administratie)
print("Nieuwe records:", eerste_publicatie)

Nieuwe records: {'geselecteerd': 20, 'bevestigd': 20, 'overgeslagen': 0}


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voer nu dezelfde handeling nogmaals uit, met dezelfde selectie en dezelfde verzendadministratie. Zonder wijzigingen verwacht je nul verstuurde en twintig overgeslagen records. Deze controle blijft ook geldig nadat je de notebookkernel opnieuw hebt gestart, zolang de verzendadministratie bewaard is.

</div>

In [11]:
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: ")) as verbinding:
    herhaling = publiceer_selectie(verbinding, topic_metingen, selectie, administratie)
assert herhaling == {"geselecteerd": len(selectie), "bevestigd": 0, "overgeslagen": len(selectie)}
print("Zelfde selectie opnieuw aangeboden:", herhaling)

Zelfde selectie opnieuw aangeboden: {'geselecteerd': 20, 'bevestigd': 0, 'overgeslagen': 20}


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Herpubliceer nu bewust dezelfde selectie met `opnieuw=True`. Je verwacht twintig bevestigingen, ook als deze records al geregistreerd waren. De code geeft de records geen nieuwe record_id. Een ontvanger die bij de eerste publicatie ontbrak, kan ze daardoor alsnog verwerken.

</div>

In [12]:
with MqttSessie(serveradres, ca, getpass("MQTT-wachtwoord van eigenaar: ")) as verbinding:
    herpublicatie = publiceer_selectie(verbinding, topic_metingen, selectie, administratie, opnieuw=True)
assert herpublicatie["bevestigd"] == len(selectie)
print("Bewuste herpublicatie:", herpublicatie)

Bewuste herpublicatie: {'geselecteerd': 20, 'bevestigd': 20, 'overgeslagen': 0}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Interpretatie</strong><br>Een tijdelijk abonnement ontvangt berichten terwijl het actief is. De publicaties gebruiken <code>retain=False</code>. Een nieuwe ontvanger krijgt dus niet automatisch de eerdere selectie. Ook een retained bericht is geen volledig meetarchief: per topic blijft dan alleen de laatste retained inhoud beschikbaar. Start in notebook 05 eerst de ontvangers en herpubliceer daarna de beginsets.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 6.** Noteer de drie tellingen. Waarom kunnen twintig brokerbevestigingen samengaan met nul opgeslagen records bij je partner?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 6):**

Voeg hier je antwoord in. Gebruik jouw eigen tellingen en benoem welke bevestiging je daadwerkelijk hebt gezien.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 6**

Bij een eerste uitvoering zijn de bevestigingen 20, 0 en 20. Bij hervatten kan de eerste telling ook 0 zijn. Een partner die nog niet verbonden was, ontvangt deze niet-bewaarde berichten niet. De PUBACK bevestigt de overdracht naar de broker, niet de opslag bij de partner. Herpublicatie na het starten van de ontvanger herstelt gemiste records.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip: de volledige beginset</strong><br>Voor een volledige overdracht kan <code>lees_metingen(..., limiet=None)</code> alle records achtereen leveren, zonder ze eerst in een lijst te laden. Geef die iterator aan <code>publiceer_selectie(..., opnieuw=True, voortgang=print)</code>. Dat gebeurt pas wanneer de ontvangers in notebook 05 klaarstaan. Twintig proefrecords zijn geen volledige beginset. Voor de eerste vier weken zijn de bronaantallen 80.028 en 146.649.</div>

## 8. Resultaat bewaren en afsluiten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 04, daarna Source Control)

Het inleverbestand bewaart wat je hebt gecontroleerd: ontvangen oefenwaarde, één teruggelezen bronidentificatie en publicatietellingen. Het bevat geen wachtwoord of serveradres. De verzendadministratie is een werkbestand en blijft buiten Git.

Voer de controlecel uit nadat de ontvangst en de drie publicatiestappen geslaagd zijn. Een groen resultaat zegt hier iets over de uitwisseling met je eigen broker. De opslag en volledigheid bij een partner worden afzonderlijk gecontroleerd in de volgende oefening.

</div>

In [13]:
assert ontvangen_oefening["payload"] == oefenbericht
assert ontvangen_aanpassing["payload"] == aangepast
assert aangepast["waarde"] != oefenbericht["waarde"]
assert ontvangen_meting["payload"] == eerste
assert herhaling["bevestigd"] == 0 and herpublicatie["bevestigd"] == len(selectie)
verslag = {
    "deelnemerscode": deelnemer, "branch": branch, "rol": rol,
    "getest_op_utc": datetime.now(timezone.utc).isoformat(),
    "topic": topic_metingen, "bron": eerste["bron"], "record_id": eerste["record_id"],
    "ontvangen_oefenwaarde": ontvangen_aanpassing["payload"]["waarde"],
    "heen_en_terug": "geslaagd", "nieuwe_records": eerste_publicatie,
    "herhaling": herhaling, "herpublicatie": herpublicatie,
}
inleverpad = hoofdmap / "gegevens/inlevering/04_publicatiecontrole.json"
inleverpad.parent.mkdir(parents=True, exist_ok=True)
inleverpad.write_text(json.dumps(verslag, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Bewaard:", inleverpad.relative_to(hoofdmap))

Bewaard: gegevens/inlevering/04_publicatiecontrole.json


<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Let op: stop je Codespace</strong><br>Een geopende Codespace blijft rekentijd gebruiken. Sla je werk op, commit en push vóór je de Codespace stopt. De Droplet heeft afzonderlijke kosten. Sluit of verwijder die alleen volgens de afspraak met de docent.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

1. Vul je antwoorden aan met jouw uitvoer. Kies **File → Save All**.
2. Open **Source Control**. Selecteer notebook 04 en `gegevens/inlevering/04_publicatiecontrole.json` met het **+**-teken. Controleer dat geen wachtwoord in een cel of uitvoer staat.
3. Commit met bijvoorbeeld **MQTT-publicatie en herpublicatie gecontroleerd** en kies **Push**. Controleer op GitHub jouw branch en beide bestanden.
4. Bewaar bij het verwijderen van een Codespace eerst je werkdatabase en `publicaties_04.db` apart op je laptop. De verzendadministratie is na deze cellen gesloten. Bij verlies ervan kan de code dezelfde records opnieuw versturen. De ontvanger moet herhalingen kunnen herkennen.
5. Je kunt de broker na de oefening handmatig stoppen met onderstaande opdracht in de SSH-terminal. Spreek dat af wanneer een partner jouw broker gebruikt. Start hem bij de volgende oefening weer handmatig.
6. Kies via [github.com/codespaces](https://github.com/codespaces) **Stop codespace**. Hervat later dezelfde Codespace.

</div>

```bash
sudo systemctl stop mosquitto
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [ ] Ik kan publisher, subscriber, broker, topic en payload onderscheiden.
- [ ] Ik heb een JSON-waarde aangepast en de ontvangen wijziging gezien.
- [ ] Ik heb een echte bronrij teruggevonden en via MQTT teruggelezen.
- [ ] Ik heb een topicfilter met `+` geprobeerd en verklaard.
- [ ] Ik kan nieuwe publicatie, herhaling en bewuste herpublicatie verklaren.
- [ ] Mijn antwoorden en controleverslag staan op mijn eigen GitHub-branch.
- [ ] Mijn werkbestanden zijn bewaard en mijn Codespace is gestopt.

</div>

## Hulp bij veelvoorkomende problemen

| Probleem | Controle en herstel |
|---|---|
| Werkbestand ontbreekt | Hervat notebook 02 voor de database of notebook 03 voor server.json en ca.crt. Gebruik geen lege vervangbestanden. |
| Verkeerde rol of ontbrekende meetkolom | Controleer je rol uit notebook 01 en de gekozen werkdatabase. VBW hoort bij gewichten, VFI bij voederopname. |
| Hokmetadata ontbreken | Controleer of je werkdatabase de tabel hokken uit notebook 02 bevat, met de juiste hokcodes. |
| Connection refused of time-out | Controleer serveradres, poort 8883 en of Mosquitto handmatig is gestart. Controleer de firewall volgens notebook 03. |
| Certificaatfout | Controleer het serveradres en het via SSH gekopieerde CA-bestand. Bij een verlopen certificaat vraag je hulp. Schakel certificaatcontrole niet uit. |
| Aanmelding geweigerd | Gebruik het MQTT-wachtwoord van eigenaar, niet het SSH- of API-wachtwoord. |
| Publicatie geweigerd | Controleer de deelnemerscode in het topic en in het rechtenbestand uit notebook 03. Het account lezer heeft geen schrijfrechten. |
| Geen ontvangst | Controleer of abonnement en topic overeenkomen. De ontvangende cel moet actief zijn tijdens het publiceren. |
| Alles wordt overgeslagen | De administratie kent deze records al. Gebruik de herpublicatiecel om ze bewust nogmaals te versturen. |
| Eerder record of metadata gewijzigd | De oefening deelt toevoegingen. Bewaar je bestanden en bespreek de wijziging. Verwijder de administratie niet om de melding te omzeilen. |
| Onderbroken publicatie | Herstel de verbinding en bied de selectie opnieuw aan. Enkele berichten kunnen herhaald worden. Voor ontbrekende historie gebruik je bewuste herpublicatie. |
| NameError na kernelherstart | Voer de voorgaande cellen opnieuw uit. Variabelen bestaan alleen tijdens de huidige kernelsessie. |

## Technische bronnen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [Eclipse Paho: Python-client, callbacks en publicatiebevestiging](https://eclipse.dev/paho/files/paho.mqtt.python/html/client.html)
- [Eclipse Mosquitto: topics, abonnementen, QoS en retained berichten](https://mosquitto.org/man/mqtt-7.html)
- [Python: JSON-omzetting](https://docs.python.org/3/library/json.html)

De verbindingscode gebruikt de Paho-callbackinterface versie 2 en MQTT 5. De technische documentatie is gecontroleerd op 18 november 2026.

</div>